# Build a Foundry Agent that calls a custom Python tool via a typed schema

The customer-support team is testing an LLM chat surface and the early transcripts include the model hallucinating order IDs and inventing shipment dates that look real enough to fool a junior agent. The product owner kills the rollout until the agent can pull real order data instead of inventing it. You have one session to wire a Foundry Agent with a typed function-calling tool against your own `get_order_status`, run a two-turn conversation that exercises one valid and one invalid order ID, and confirm via the run trace that the tool fired exactly once on the valid path and that the invalid path surfaces a graceful user-facing apology instead of a hallucination.

Two big mental-model fixes this lab forces:

- The agent does NOT execute your Python function. It emits a tool-call directive in JSON. Your application loop interprets the directive and runs the function locally.
- The agent's `function.arguments` field is a JSON STRING, not a parsed dict. You must `json.loads` it before passing to `**kwargs`.

Both are exam-relevant and both are how students first get stuck.

**Time.** Plan for about 50 minutes hands-on. Foundry hub plus project provisioning is 5 to 8 minutes; agent create is fast; the dispatch loop polls. Budget up to 90 if you debug the tool result shape. The cleanup cell at the bottom tears every Azure resource down.

**Cost.** This lab costs under five cents end-to-end. GPT-4o-mini for two conversation turns at ~500 input + 150 output tokens each works out to fractions of a cent. The Foundry Agent Service orchestration layer is free.

This lab maps to AI-103 Domain 2: Implement generative AI and agentic solutions (33% of exam weight), specifically the function-calling task statement.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Azure SDKs. azure-ai-projects 2.0 (March 2026)
# absorbed azure-ai-agents; never install azure-ai-agents directly per the cert
# YAML risks list. The deprecated Assistants API (client.beta.assistants)
# retires 2026-08-26.

!pip install --quiet labrun-checks[azure]==0.7.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT section 12.

import atexit
import getpass
import json
import time

from azure.identity import DefaultAzureCredential
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError, ClientAuthenticationError
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from azure.ai.projects import AIProjectClient

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)
from labrun_checks.adapters.azure import AzureCleanupAdapter

LAB_ID = "agent-with-function-calling"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4omini"
AGENT_NAME = f"labrun-{LAB_ID}-agent"
MODEL_NAME = "gpt-4o-mini"
MODEL_VERSION = "2024-07-18"
MODEL_CAPACITY_TPM = 30

SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
PROJECT_ENDPOINT = None
AGENT_ID = None
THREAD_ID = None
AZURE_CREDENTIAL = None

# Pricing for the wallet check.
PRICE_PER_1M_INPUT_USD = 0.15
PRICE_PER_1M_OUTPUT_USD = 0.60

# Conversation state for the validators.
VALID_RUN_TRACE = None
INVALID_RUN_TRACE = None
VALID_FINAL_MESSAGE = None
INVALID_FINAL_MESSAGE = None

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials via
# DefaultAzureCredential per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. Foundry hosted agents pin {REGION}.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION, "resource_group": RESOURCE_GROUP}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# Reverse-creation order: agent first (deleting the agent also garbage-collects
# its conversation threads), then the deployment, project, hub, resource group.
# No critical resources; GPT-4o-mini Standard and the agent service do not bill
# hourly at zero traffic.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="foundry_agent",
        id=AGENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml agent delete --resource-group {RESOURCE_GROUP} "
            f"--workspace-name {PROJECT_NAME} --agent-id <agent-id-resolved-at-create>"
        ),
        extra={"project_endpoint": PROJECT_ENDPOINT},
    ),
    CleanupResource(
        type="aoai_deployment",
        id=DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {DEPLOYMENT_NAME}"
        ),
        extra={"account_name": AOAI_ACCOUNT_NAME},
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print(f"Run the cleanup cell first, or: az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Provision Foundry, deploy GPT-4o-mini, and create the agent with a typed tool schema

Three layers stack here:

1. **Foundry hub plus project.** Same pattern as Lab 01. Hubs are the umbrella; projects are the working unit each app uses.
2. **GPT-4o-mini deployment.** Same pattern as Lab 01.
3. **Foundry Agent on the Responses API path.** Use `client.agents.create_agent(...)` from `azure.ai.projects` 2.0. The agent is created against the project's endpoint, references the deployment by name, and carries a tool schema.

**Anti-pattern: the deprecated Assistants surface.** Older tutorials show `client.beta.assistants.create(...)`. That path retires 2026-08-26. The validator inspects the agent's surface metadata and rejects anything that traces back to Assistants. Always use `client.agents.create_agent` from `azure.ai.projects`.

**Tool schema shape.** JSON Schema describing one function:

- `type="function"`
- `function.name="get_order_status"`
- `function.description="..."`
- `function.parameters.type="object"`
- `function.parameters.properties.order_id.type="string"`
- `function.parameters.required=["order_id"]`

The checkpoint walks every field. A missing `required` array is the most common miss because students assume the model will figure it out.

In [ ]:
# NBVAL_SKIP
# Task 1: Provision RG, hub, project, GPT-4o-mini deployment, then create the
# agent with a typed function tool schema.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
).result()

project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id},
}
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
project_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
).result()
PROJECT_ENDPOINT = project_workspace.properties.discovery_url

for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")

deployment_payload = {
    "sku": {"name": "Standard", "capacity": MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": MODEL_NAME, "version": MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Watching the model deployment go through Succeeded purgatory...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, DEPLOYMENT_NAME, deployment_payload,
).result()
print(f"GPT-4o-mini deployment ready: {DEPLOYMENT_NAME}")

# Authenticate against the project and create the agent on the Responses API
# path. Never use client.beta.assistants; that path retires 2026-08-26.
cred = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cred)

tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": (
                "Look up the status of a customer order by order ID. Returns the "
                "status string, shipment date, and last-updated timestamp."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID to look up, e.g. ORD-002.",
                    },
                },
                "required": ["order_id"],
            },
        },
    }
]

agent_instructions = (
    "You are a customer-support agent. When the user asks about an order, call "
    "get_order_status with the order_id they mentioned. If the tool returns an "
    "error, apologise and tell the user you could not find that order. Never "
    "invent order details that did not come from the tool."
)

# YOUR CODE: Create the agent with project_client.agents.create_agent(
# model=DEPLOYMENT_NAME, name=AGENT_NAME, instructions=agent_instructions,
# tools=tool_schema, metadata={LAB_TAG_KEY: LAB_TAG_VALUE}).
# Store the result in agent.

AGENT_ID = agent.id
print(f"Agent created: {AGENT_NAME} (id: {AGENT_ID})")

# Update the cleanup manifest now that AGENT_ID is resolved.
for r in CLEANUP_MANIFEST:
    if r.type == "foundry_agent" and "<agent-id-resolved-at-create>" in (r.cli_delete_command or ""):
        r.id = AGENT_ID
        r.cli_delete_command = r.cli_delete_command.replace(
            "<agent-id-resolved-at-create>", AGENT_ID,
        )

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Agent exists on the Responses API path, references the
# deployment, and the tool schema has the expected shape (type=function,
# function.name=get_order_status, parameters.required=["order_id"],
# parameters.properties.order_id.type="string"). Rejects anything traceable
# to the deprecated Assistants surface.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        if not AGENT_ID:
            return CheckpointResult(
                status="fail",
                error_reason="AGENT_ID is unset. Re-run Task 1 to create the agent.",
            )

        client_local = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
        try:
            agent_now = client_local.agents.get_agent(AGENT_ID)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=f"Agent {AGENT_ID} could not be fetched via client.agents.get_agent.",
            )

        # Reject deprecated Assistants surface. Foundry 2.0 agents expose a
        # surface marker; anything that looks like the beta.assistants path is
        # the wrong shape.
        cls_name = type(agent_now).__name__
        if "assistant" in cls_name.lower() and "agent" not in cls_name.lower():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Agent class is {cls_name!r}; this looks like the deprecated "
                    f"Assistants surface. Use client.agents.create_agent from "
                    f"azure.ai.projects, not client.beta.assistants."
                ),
            )

        model_ref = getattr(agent_now, "model", None) or ""
        if model_ref != DEPLOYMENT_NAME:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Agent references model={model_ref!r}, expected {DEPLOYMENT_NAME!r}. "
                    f"Pass deployment name, not raw model name."
                ),
            )

        tools = getattr(agent_now, "tools", None) or []
        function_tools = []
        for t in tools:
            t_dict = t if isinstance(t, dict) else getattr(t, "as_dict", lambda: {})()
            if t_dict.get("type") == "function":
                function_tools.append(t_dict)
        if not function_tools:
            return CheckpointResult(
                status="fail",
                error_reason="Agent has no function-type tools. Did the tools= argument land?",
            )

        func = function_tools[0].get("function", {})
        if func.get("name") != "get_order_status":
            return CheckpointResult(
                status="fail",
                error_reason=f"Tool function.name is {func.get('name')!r}, expected 'get_order_status'.",
            )
        params = func.get("parameters", {}) or {}
        if params.get("type") != "object":
            return CheckpointResult(
                status="fail",
                error_reason=f"parameters.type is {params.get('type')!r}, expected 'object'.",
            )
        props = params.get("properties") or {}
        order_id_prop = props.get("order_id") or {}
        if order_id_prop.get("type") != "string":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"parameters.properties.order_id.type is {order_id_prop.get('type')!r}, "
                    f"expected 'string'."
                ),
            )
        if "order_id" not in (params.get("required") or []):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "parameters.required does not include 'order_id'. Without required, the model "
                    "can call the tool with no arguments and your dispatch loop crashes."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names the exact field that is off: `function.name`, `parameters.type`, `parameters.properties.order_id.type`, or `parameters.required`. A common miss: leaving `required` empty, which lets the model call the tool with no arguments.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `project_client.agents.create_agent(model=DEPLOYMENT_NAME, name=AGENT_NAME, instructions=agent_instructions, tools=tool_schema, metadata={LAB_TAG_KEY: LAB_TAG_VALUE})`. `model` is the deployment name, not the raw `gpt-4o-mini`. The `tool_schema` variable already has the JSON Schema set up in the cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution:

```python
agent = project_client.agents.create_agent(
    model=DEPLOYMENT_NAME,
    name=AGENT_NAME,
    instructions=agent_instructions,
    tools=tool_schema,
    metadata={LAB_TAG_KEY: LAB_TAG_VALUE},
)
```

</details>

**Wallet check.** Still at $0.00. Creating the agent is a control-plane call; the token meter has not started. The hub, project, and Standard deployment carry no hourly fee. Coffee is well ahead.

## Task 2: Dispatch the tool on a valid order ID and confirm the run trace records exactly one call

The application is the runtime. Your local Python `get_order_status` function lives in this notebook process; the agent does not have direct access to it. The dispatch loop is documented:

1. Create a conversation thread: `thread = project_client.agents.create_thread()`.
2. Post a user message: `project_client.agents.create_message(thread_id=thread.id, role="user", content="What is the status of order ORD-002?")`.
3. Start a run: `run = project_client.agents.create_run(thread_id=thread.id, agent_id=AGENT_ID)`.
4. Poll until `run.status` is `requires_action`, `completed`, or `failed`. Use a backoff capped at 30 seconds total.
5. While `requires_action`, read `run.required_action.submit_tool_outputs.tool_calls`, dispatch each one against the local function, and call `project_client.agents.submit_tool_outputs(thread_id=thread.id, run_id=run.id, tool_outputs=[{"tool_call_id": ..., "output": <JSON string>}])`. Repeat the poll-and-dispatch loop until the run is `completed`.

Two big traps:

- `function.arguments` is a JSON STRING. `json.loads` it before passing to your function.
- The `output` field on `submit_tool_outputs` must be a STRING (typically a JSON-serialised dict). Passing a Python dict returns a 400.

In [ ]:
# NBVAL_SKIP
# Task 2: Run a turn with a valid order ID. Implement get_order_status as a
# local function with deterministic mock data, then drive the dispatch loop.

MOCK_ORDERS = {
    "ORD-001": {"status": "shipped", "shipment_date": "2026-05-03", "last_updated": "2026-05-03T14:02:00Z"},
    "ORD-002": {"status": "in_transit", "shipment_date": "2026-05-09", "last_updated": "2026-05-12T08:15:00Z"},
    "ORD-003": {"status": "delivered", "shipment_date": "2026-04-28", "last_updated": "2026-04-29T17:40:00Z"},
}


class OrderNotFoundError(Exception):
    pass


def get_order_status(order_id: str) -> dict:
    if order_id not in MOCK_ORDERS:
        raise OrderNotFoundError(order_id)
    return {"order_id": order_id, **MOCK_ORDERS[order_id]}


def dispatch_tool_call(tool_call) -> str:
    """Translate one tool-call directive into a local function call and return
    the tool_outputs string the agent expects (always JSON-serialised)."""
    name = tool_call.function.name
    args_str = tool_call.function.arguments or "{}"
    args = json.loads(args_str)
    if name != "get_order_status":
        return json.dumps({"error": "unknown_tool", "tool": name})
    try:
        result = get_order_status(**args)
        return json.dumps(result)
    except OrderNotFoundError as exc:
        return json.dumps({"error": "order_not_found", "order_id": str(exc)})


def poll_run(thread_id: str, run_id: str, ceiling_seconds: int = 30):
    delay = 1.0
    elapsed = 0.0
    while elapsed < ceiling_seconds:
        run_now = project_client.agents.get_run(thread_id=thread_id, run_id=run_id)
        if run_now.status in ("requires_action", "completed", "failed", "cancelled", "expired"):
            return run_now
        time.sleep(delay)
        elapsed += delay
        delay = min(delay * 1.5, 4.0)
    return project_client.agents.get_run(thread_id=thread_id, run_id=run_id)


def drive_conversation_turn(thread_id: str, user_message: str):
    project_client.agents.create_message(thread_id=thread_id, role="user", content=user_message)
    run = project_client.agents.create_run(thread_id=thread_id, agent_id=AGENT_ID)
    while True:
        run = poll_run(thread_id, run.id)
        if run.status == "requires_action":
            tool_calls = run.required_action.submit_tool_outputs.tool_calls
            print("Dispatching get_order_status with the agent's chosen arguments...")
            outputs = []
            for tc in tool_calls:
                output_str = dispatch_tool_call(tc)
                outputs.append({"tool_call_id": tc.id, "output": output_str})
            print("Feeding the tool result back to the agent for its natural-language reply...")
            # YOUR CODE: Submit the tool outputs back to the run using
            # project_client.agents.submit_tool_outputs(thread_id=thread_id,
            # run_id=run.id, tool_outputs=outputs).
            continue
        return run


thread = project_client.agents.create_thread()
THREAD_ID = thread.id
print(f"Conversation thread created: {THREAD_ID}")

print()
print("Asking the agent to think about whether to call the tool (valid order)...")
VALID_RUN_TRACE = drive_conversation_turn(THREAD_ID, "What is the status of order ORD-002?")

# Read the most recent assistant message in the thread.
messages = list(project_client.agents.list_messages(thread_id=THREAD_ID))
for msg in messages:
    if msg.role == "assistant":
        VALID_FINAL_MESSAGE = msg.content[0].text.value if msg.content else ""
        break
print()
print("Assistant reply:")
print(VALID_FINAL_MESSAGE)

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: The valid-order turn's run trace shows exactly one tool call
# to get_order_status with order_id=ORD-002, and the final assistant message
# references a non-empty status field from the mock data (shipped / in_transit
# / delivered).

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        if VALID_RUN_TRACE is None or THREAD_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="VALID_RUN_TRACE or THREAD_ID is unset. Re-run Task 2 first.",
            )
        if VALID_RUN_TRACE.status != "completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Valid-order run status is {VALID_RUN_TRACE.status!r}, expected 'completed'. "
                    f"Did the dispatch loop submit tool outputs?"
                ),
            )

        client_local = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
        steps = list(client_local.agents.list_run_steps(thread_id=THREAD_ID, run_id=VALID_RUN_TRACE.id))
        tool_call_steps = []
        for step in steps:
            step_details = getattr(step, "step_details", None)
            calls = getattr(step_details, "tool_calls", None) if step_details else None
            if calls:
                for tc in calls:
                    fn = getattr(tc, "function", None)
                    if fn is not None:
                        tool_call_steps.append(fn)

        if len(tool_call_steps) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run trace has {len(tool_call_steps)} tool calls on the valid-order turn, "
                    f"expected exactly 1. If >1, the agent looped (often because the tool output "
                    f"was malformed). If 0, the agent did not call the tool."
                ),
            )
        only_call = tool_call_steps[0]
        if getattr(only_call, "name", None) != "get_order_status":
            return CheckpointResult(
                status="fail",
                error_reason=f"Tool call name is {getattr(only_call, 'name', None)!r}, expected 'get_order_status'.",
            )
        try:
            args = json.loads(getattr(only_call, "arguments", "") or "{}")
        except json.JSONDecodeError:
            return CheckpointResult(
                status="fail",
                error_reason="Tool call arguments did not parse as JSON.",
            )
        if args.get("order_id") != "ORD-002":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Tool call arguments were {args!r}, expected order_id=ORD-002. "
                    f"The agent picked up the wrong ID from the user message."
                ),
            )

        msg = (VALID_FINAL_MESSAGE or "").lower()
        if not any(kw in msg for kw in ("shipped", "in_transit", "in transit", "delivered")):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Final assistant message does not reference a status field from the tool result. "
                    f"Got: {VALID_FINAL_MESSAGE!r}"
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

If the run trace has >1 tool calls, the agent looped on the same call because the previous tool output was malformed (it expects a string). If the run trace has 0 tool calls, the agent answered from the instructions instead of using the tool: tighten the instructions or check that the user message names a known order ID format. If the run is still `requires_action`, the dispatch loop never submitted outputs.

</details>

<details><summary>Hint 2 (direction)</summary>

One missing call: `project_client.agents.submit_tool_outputs(thread_id=thread_id, run_id=run.id, tool_outputs=outputs)`. The `outputs` list is already correctly shaped above (each entry has `tool_call_id` and a JSON-string `output`). Without that submit call, the run never resumes.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution:

```python
project_client.agents.submit_tool_outputs(
    thread_id=thread_id, run_id=run.id, tool_outputs=outputs,
)
```

</details>

**Wallet check.** Around a tenth of a cent. One agent turn at roughly 500 input + 150 output tokens costs about $0.000075 + $0.00009 = $0.000165. The agent service orchestration is free. Total damage so far: well under a penny.

## Task 3: Run an invalid order ID and confirm the agent apologises instead of hallucinating

On the same thread (the conversation context carries over), ask about `ORD-999`. Your local `get_order_status` raises `OrderNotFoundError`; the dispatch loop catches it and submits `{"error": "order_not_found", "order_id": "ORD-999"}` as the tool result string. The agent reads the structured error in the next inference pass and produces a natural-language apology.

The validator checks three things:

1. The literal string `ORD-999` appears in the assistant message.
2. The message does NOT contain a fabricated status (`shipped`, `delivered`, `in_transit`) or an ISO date for this turn.
3. The message contains an apology or unknown-state framing (`sorry`, `could not`, `unable`, `not find`, `no record`).

The dispatch loop from Task 2 already handles the exception path; you do not need to write new code, just run the same `drive_conversation_turn` against the invalid ID.

In [ ]:
# NBVAL_SKIP
# Task 3: Same thread, invalid order ID. The dispatch loop converts
# OrderNotFoundError into a structured tool result the agent can read.

print("Asking the agent about an order ID that does not exist...")
# YOUR CODE: Call INVALID_RUN_TRACE = drive_conversation_turn(THREAD_ID,
# "And what about ORD-999?") to drive the same dispatch loop on the invalid ID.

# Read the most recent assistant message after this turn.
messages_now = list(project_client.agents.list_messages(thread_id=THREAD_ID))
for msg in messages_now:
    if msg.role == "assistant":
        INVALID_FINAL_MESSAGE = msg.content[0].text.value if msg.content else ""
        break
print()
print("Assistant reply (invalid order):")
print(INVALID_FINAL_MESSAGE)

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Invalid-order final message references ORD-999, lacks any
# fabricated status or ISO date, and contains an apology / unknown-state
# framing.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    import re
    try:
        if not INVALID_FINAL_MESSAGE:
            return CheckpointResult(
                status="fail",
                error_reason="INVALID_FINAL_MESSAGE is empty. Did Task 3 run to completion?",
            )
        msg = INVALID_FINAL_MESSAGE
        msg_lower = msg.lower()

        if "ord-999" not in msg_lower:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Final assistant message does not reference 'ORD-999'. Got: {msg!r}. "
                    f"The agent should mention the order ID the user asked about."
                ),
            )

        fabricated_status_terms = ["shipped", "in_transit", "in transit", "delivered"]
        present_status = [s for s in fabricated_status_terms if s in msg_lower]
        if present_status:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Final assistant message contains a status term {present_status} for an order "
                    f"the tool reported as not found. The agent is hallucinating."
                ),
            )
        if re.search(r"\b\d{4}-\d{2}-\d{2}\b", msg):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Final assistant message contains an ISO date for the invalid order. "
                    "The tool returned no date; the agent fabricated it."
                ),
            )

        apology_terms = ["sorry", "could not", "couldn't", "unable", "not find", "no record", "don't have", "do not have"]
        if not any(term in msg_lower for term in apology_terms):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Final assistant message lacks an apology or unknown-state framing. Got: {msg!r}. "
                    f"Expected one of: sorry, could not, unable, not find, no record."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

If the agent invented a status, the structured error output is not making it into the agent's context. Confirm the dispatch loop is calling `submit_tool_outputs` with `output=json.dumps({"error": "order_not_found", "order_id": "ORD-999"})`, not skipping it or sending an empty string. If the message lacks ORD-999, the agent dropped the ID; usually the instructions need to be tightened.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `INVALID_RUN_TRACE = drive_conversation_turn(THREAD_ID, "And what about ORD-999?")`. The dispatch loop already wraps `get_order_status` in try/except and converts `OrderNotFoundError` into the structured error dict; you do not need to write new dispatch code for this task.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution:

```python
INVALID_RUN_TRACE = drive_conversation_turn(THREAD_ID, "And what about ORD-999?")
```

</details>

**Wallet check.** Around two-tenths of a cent. The second turn is roughly the same cost as the first. Total damage so far: still under half a penny.

## Task 4: Inspect the run trace and confirm tool-call lineage on the valid-order turn

The Foundry Agent Service captures every run step in order: `tool_call`, `tool_output`, then `message_creation`. This lineage is what powers Foundry's tracing tab in production and what auditors inspect when a customer disputes an action the agent took.

Build it in this order:

1. Call `project_client.agents.list_run_steps(thread_id=THREAD_ID, run_id=VALID_RUN_TRACE.id)` to get the ordered steps.
2. Walk the steps and confirm a `tool_call` step appears before a `tool_output` step (also called `tool_calls` step output in some surfaces), followed by a `message_creation` step.
3. Print the lineage so the audit story is concrete.

In [ ]:
# NBVAL_SKIP
# Task 4: Walk the run trace and confirm the tool-call lineage.

print("Reading the agent's run steps to confirm the lineage is intact...")
# YOUR CODE: Call run_steps = list(project_client.agents.list_run_steps(
# thread_id=THREAD_ID, run_id=VALID_RUN_TRACE.id)).

STEP_TYPES = []
for step in run_steps:
    step_type = getattr(step, "type", None) or getattr(step, "step_type", None)
    STEP_TYPES.append(step_type)
    print(f"  step {step.id}: type={step_type}")

print()
print(f"Step types in run trace: {STEP_TYPES}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: The valid-order run trace contains a tool_calls step and a
# message_creation step in that order, with the tool call carrying the
# correct argument JSON. The Foundry Agent Service combines tool-call
# emission and tool-output ingestion into a single step type called
# 'tool_calls' (the step holds an array of tool_call entries, each with the
# canonical function arguments).

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        if VALID_RUN_TRACE is None or THREAD_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="VALID_RUN_TRACE or THREAD_ID is unset. Re-run Tasks 2 and 4.",
            )
        client_local = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
        steps = list(client_local.agents.list_run_steps(thread_id=THREAD_ID, run_id=VALID_RUN_TRACE.id))
        if not steps:
            return CheckpointResult(
                status="fail",
                error_reason="list_run_steps returned no steps for the valid-order run.",
            )
        step_types = [getattr(s, "type", None) or getattr(s, "step_type", None) for s in steps]

        # Steps return newest-first by default; reverse for chronological reading.
        ordered = list(reversed(step_types))
        try:
            tool_idx = ordered.index("tool_calls")
        except ValueError:
            return CheckpointResult(
                status="fail",
                error_reason=f"Run trace has no 'tool_calls' step. Step types found: {ordered}",
            )
        try:
            msg_idx = ordered.index("message_creation")
        except ValueError:
            return CheckpointResult(
                status="fail",
                error_reason=f"Run trace has no 'message_creation' step. Step types found: {ordered}",
            )
        if not (tool_idx < msg_idx):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run trace ordering is wrong. tool_calls index {tool_idx} should precede "
                    f"message_creation index {msg_idx}. Steps: {ordered}"
                ),
            )

        # Confirm the tool call argument JSON is intact.
        tool_call_step = next(
            s for s in steps
            if (getattr(s, "type", None) or getattr(s, "step_type", None)) == "tool_calls"
        )
        details = getattr(tool_call_step, "step_details", None)
        calls = getattr(details, "tool_calls", None) if details else None
        if not calls:
            return CheckpointResult(
                status="fail",
                error_reason="tool_calls step has no tool_calls array.",
            )
        fn = getattr(calls[0], "function", None)
        if fn is None:
            return CheckpointResult(
                status="fail",
                error_reason="tool_call entry has no function field.",
            )
        if getattr(fn, "name", None) != "get_order_status":
            return CheckpointResult(
                status="fail",
                error_reason=f"tool_call.function.name is {getattr(fn, 'name', None)!r}, expected 'get_order_status'.",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The validator wants two step types: `tool_calls` followed by `message_creation`. The Foundry surface returns steps newest-first; if you read the list in raw order it looks like `message_creation` came first.

</details>

<details><summary>Hint 2 (direction)</summary>

One line: `run_steps = list(project_client.agents.list_run_steps(thread_id=THREAD_ID, run_id=VALID_RUN_TRACE.id))`. The step types are on `step.type` (or `step.step_type` on older SDKs); the checkpoint accepts either attribute name. For chronological reading, reverse the list.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution:

```python
run_steps = list(project_client.agents.list_run_steps(
    thread_id=THREAD_ID, run_id=VALID_RUN_TRACE.id,
))
```

</details>

**Wallet check.** Still around two-tenths of a cent. `list_run_steps` is a control-plane read and free. Total damage for the session is under half a penny.

## Cleanup

Time to tear it all down. The cell runs the manifest in reverse-creation order: agent first (deleting the agent also garbage-collects its conversation threads), then the deployment, then project, hub, resource group. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST.

import sys

for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

# Patch resource.extra with constants resolved during the lab so the
# Azure adapter sees account_name, service_name, and project_endpoint
# at cleanup time (manifest is built before those constants are set).
for r in CLEANUP_MANIFEST:
    if r.type in ("aoai_deployment", "aoai_rai_policy") and AOAI_ACCOUNT_NAME:
        r.extra = {"account_name": AOAI_ACCOUNT_NAME}
    elif r.type == "search_index" and "SEARCH_SERVICE_NAME" in globals() and SEARCH_SERVICE_NAME:
        r.extra = {"service_name": SEARCH_SERVICE_NAME}
    elif r.type == "foundry_agent" and "PROJECT_ENDPOINT" in globals() and PROJECT_ENDPOINT:
        r.extra = {"project_endpoint": PROJECT_ENDPOINT}

result = run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: under half a penny.** Two agent turns at GPT-4o-mini rates barely move the meter. The Foundry hub, project, agent, and Standard deployment carry no hourly fee at zero traffic; the resource group delete is the safety net. Check Azure Cost Management in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. The agent did not execute the Python function for you. It returned a directive that said "call this tool with these arguments" and waited. Walk through why function-calling is architected this way instead of letting the agent execute code directly, and what would change in your security review if the agent could.

2. Your team is adding a second tool to this agent, `cancel_order(order_id: str)`. What changes in the tool schema, the dispatch loop, and the audit story? When would you choose to layer this as a separate agent versus another tool on the same agent?

## Exam-Style Practice

**Question 1 (Domain 2, tool dispatch architecture):**

You build a Foundry Agent with a `get_order_status(order_id)` tool. The agent runs and a user asks about an order. The run's status enters `requires_action`. Which application code path resumes the run?

A. `client.agents.continue_run(thread_id, run_id)` automatically dispatches the tool against the local Python function the agent was registered with.

B. The application reads `run.required_action.submit_tool_outputs.tool_calls`, executes each tool locally, then calls `client.agents.submit_tool_outputs(thread_id, run_id, tool_outputs=[...])` with a list of `{tool_call_id, output}` dicts.

C. The Foundry Agent Service executes the registered tool function on Azure and the application just polls for the final response.

D. The application calls `client.agents.cancel_run(run_id)` and restarts the conversation; tool execution requires a synchronous run path that Responses API does not support.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: there is no `continue_run` that knows how to dispatch tools. The agent never has direct access to the application's Python functions. The application is the runtime.
- B is correct: this is the documented function-calling loop. The agent emits a tool-call directive in `required_action`; the application's job is to dispatch, capture the result, and submit it back via `submit_tool_outputs`. The run resumes automatically once the tool outputs land.
- C is wrong: Foundry Agent Service does NOT execute arbitrary application code. The student's local function lives only in the student's process. Microsoft does not run user-supplied Python on the agent runtime.
- D is wrong: Responses API supports tool calling natively; cancellation is not the right path. The run is in a normal state requiring application action.

</details>

**Question 2 (Domain 2, function calling vs MCP vs Foundry Tools):**

A team is building an agent that calls four internal microservices (orders, returns, refunds, shipping). Each microservice has its own auth, deploys independently, and is owned by a separate team. Which integration approach minimises per-microservice integration cost?

A. Define four `function`-type tool schemas in the agent, with the application dispatching each via a local Python adapter for each service.

B. Wrap each microservice as an MCP server and connect the agent to the four MCP servers via the MCP connector with OAuth passthrough.

C. Use Foundry Tools' built-in Azure Translator and Document Intelligence connectors as proxies for the four microservices.

D. Deploy a single REST gateway in front of the four microservices and register it as one `function` tool that takes a `service` argument.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: the application becomes the integration hub for all four services, requiring per-service auth, per-service error handling, and changes every time a microservice changes. High per-microservice cost.
- B is correct: MCP is purpose-built for tool integration with auth passthrough. Each microservice team ships and owns their MCP server. The agent connects via the MCP connector once and inherits the four toolsets. Each team owns its own surface.
- C is wrong: Foundry Tools are pre-built connectors for Azure cognitive surfaces, not internal microservices. They do not adapt to arbitrary REST endpoints.
- D is wrong: this introduces a new gateway service the team must own. Not minimum cost. MCP is the documented Microsoft pattern for this exact problem.

</details>